# Data Preparation — Customer Purchase Propensity

Sau EDA, notebook này chuẩn bị bảng household–snapshot phục vụ bước mô hình hóa. Tại cutoff `s`, lịch sử kết thúc ở `s`; target bằng 1 khi hộ có **bất kỳ basket** trong `(s, s+7]`, bao gồm fuel, miscellaneous và basket zero-sales.

Đơn vị là một household tại một cutoff. Snapshot được lập hằng tuần, từ ngày 180 đến 698, chỉ cho hộ đã xuất hiện đến cutoff. Lịch sử 180 ngày là độ dài cửa sổ theo thời gian của nguồn, **không yêu cầu** mỗi hộ đã được quan sát đủ 180 ngày.

Phần này chuẩn bị và kiểm tra dữ liệu; chưa huấn luyện mô hình. Tập test đã từng được xem trong quá trình làm dự án, vì vậy không phải holdout hoàn toàn chưa từng xem.

## 1. Nạp giao dịch và kiểm tra đơn vị basket

Chỉ dùng các cột cần thiết cho bộ biến giao dịch ban đầu. `QUANTITY` không được tổng hợp; demographic và marketing chưa đưa vào vì thời điểm sẵn có theo từng cutoff chưa được xác minh. Giữ mọi basket, không lọc doanh thu hay department.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA = ROOT / 'sample_data' / 'dunnhumby' / 'transaction_data.csv'
OUT = ROOT / 'artifacts' / 'data_preparation'
tx = pd.read_csv(DATA, usecols=['household_key', 'BASKET_ID', 'DAY', 'SALES_VALUE'])
assert not tx[['household_key', 'BASKET_ID', 'DAY', 'SALES_VALUE']].isna().any().any()
assert tx.DAY.min() >= 1
basket_scope = tx.groupby('BASKET_ID').agg(households=('household_key','nunique'), days=('DAY','nunique'))
assert basket_scope.households.eq(1).all() and basket_scope.days.eq(1).all()
baskets = (tx.groupby(['household_key','BASKET_ID','DAY'], as_index=False, sort=False)
           .agg(sales=('SALES_VALUE','sum'), line_items=('SALES_VALUE','size')))
assert not baskets.duplicated(['household_key','BASKET_ID','DAY']).any()
print(f'{len(tx):,} dòng giao dịch → {len(baskets):,} basket → {baskets.household_key.nunique():,} household')
print('DAY:', tx.DAY.min(), 'đến', tx.DAY.max())

2,595,732 dòng giao dịch → 276,484 basket → 2,500 household
DAY: 1 đến 711


## 2. Chuyển basket thành lịch sử household–ngày

Một ngày hoạt động có ít nhất một basket; nhiều basket trong cùng ngày vẫn chỉ là một ngày. `sales` là số tiền retailer nhận được theo nguồn, không mặc định là tiền khách thực trả.

In [2]:
daily = (baskets.groupby(['household_key','DAY'], as_index=False)
         .agg(baskets=('BASKET_ID','size'), sales=('sales','sum'), line_items=('line_items','sum')))
daily['active_days'] = 1
assert not daily.duplicated(['household_key','DAY']).any()
assert daily.baskets.sum() == len(baskets)
display(daily.head())

,household_key,DAY,baskets,sales,line_items,active_days
0,1,51,1,78.66,30,1
1,1,67,1,41.10,12,1
2,1,88,1,26.90,12,1
3,1,94,1,63.43,23,1
4,1,101,1,53.45,17,1


## 3. Tạo household–snapshot theo thời gian

Với mỗi cửa sổ 7, 30, 90 và 180 ngày, tính số basket, ngày hoạt động, dòng hàng và tổng `SALES_VALUE` trong `(s-w,s]`. `recency_days` là số ngày kể từ basket gần nhất; `observed_tenure_days` là số ngày từ lần đầu quan sát. Không đưa cutoff hoặc mã household vào danh sách biến dự báo một cách mặc định.

Target sử dụng `(s,s+7]`: basket tại cutoff thuộc lịch sử, basket tại `s+7` thuộc target và basket tại `s+8` không thuộc target.

In [3]:
HORIZON = 7
SNAPSHOTS = list(range(180, 699, 7))
WINDOWS = (7, 30, 90, 180)
max_day = int(daily.DAY.max())
assert max(SNAPSHOTS) + HORIZON <= max_day
households = np.sort(daily.household_key.unique())
matrices = {}
for metric in ['baskets','sales','line_items','active_days']:
    matrices[metric] = (daily.pivot(index='household_key', columns='DAY', values=metric)
                        .reindex(index=households, columns=range(max_day+1))
                        .fillna(0).to_numpy(dtype=np.float64))
cumulative = {metric: np.cumsum(matrix, axis=1) for metric, matrix in matrices.items()}
activity = matrices['baskets'] > 0
day_axis = np.arange(max_day+1)[None, :]
first = np.where(activity, day_axis, max_day+1).min(axis=1)
last = np.maximum.accumulate(np.where(activity, day_axis, -1), axis=1)
frames = []
for s in SNAPSHOTS:
    values = {'household_key': households, 'snapshot_day': s,
              'recency_days': s-last[:,s], 'observed_tenure_days': s-first}
    for w in WINDOWS:
        for metric in matrices:
            values[f'{metric}_{w}d'] = cumulative[metric][:,s] - cumulative[metric][:,s-w]
    values['target_any_basket'] = ((cumulative['baskets'][:,s+HORIZON] - cumulative['baskets'][:,s]) > 0).astype('int8')
    frames.append(pd.DataFrame(values).loc[first <= s])
panel = pd.concat(frames, ignore_index=True)
assert not panel.duplicated(['household_key','snapshot_day']).any()
assert panel.recency_days.ge(0).all() and panel.observed_tenure_days.ge(0).all()
assert panel.target_any_basket.isin([0,1]).all()
for w in WINDOWS:
    assert panel[f'active_days_{w}d'].between(0,w).all()
    assert (panel[f'active_days_{w}d'] <= panel[f'baskets_{w}d']).all()
print('Tổng household–snapshot:', f'{len(panel):,}', '| Số cutoff:', len(SNAPSHOTS))

Tổng household–snapshot: 187,305 | Số cutoff: 75


## 4. Phân chia theo thời gian

Giữ cách chia đã dùng trong project. Hai cutoff 544 và 628 là embargo: không đưa vào các tập. Khoảng cách bảo đảm cửa sổ nhãn của tập trước kết thúc trước cutoff đầu tiên của tập sau. Lịch sử giữa các tập có thể chồng lấn, giống bối cảnh dự báo lặp cho cùng household; điều này không làm các quan sát độc lập.

| Tập | Cutoff | Cửa sổ nhãn cuối cùng |
|---|---|---|
| Train | 180–537 | đến 544 |
| Validation | 551–621 | đến 628 |
| Test | 635–698 | đến 705 |

Không sử dụng test để chọn đặc trưng hay quyết định mô hình. Tập test đã từng được xem ở các giai đoạn trước và không được coi là tập hoàn toàn chưa từng xem.

In [4]:
def assign_split(s):
    if 180 <= s <= 537: return 'train'
    if 551 <= s <= 621: return 'validation'
    if 635 <= s <= 698: return 'test'
    return 'embargo'
panel['split'] = panel.snapshot_day.map(assign_split)
assert set(panel.loc[panel.split.eq('embargo'), 'snapshot_day']) == {544, 628}
assert panel.loc[panel.split.eq('train'), 'snapshot_day'].max()+HORIZON < panel.loc[panel.split.eq('validation'), 'snapshot_day'].min()
assert panel.loc[panel.split.eq('validation'), 'snapshot_day'].max()+HORIZON < panel.loc[panel.split.eq('test'), 'snapshot_day'].min()
summary = panel.groupby('split').agg(rows=('household_key','size'), households=('household_key','nunique'),
                                      cutoff_min=('snapshot_day','min'), cutoff_max=('snapshot_day','max'),
                                      cutoffs=('snapshot_day','nunique'))
display(summary.loc[['train','validation','test','embargo']])

,rows,households,cutoff_min,cutoff_max,cutoffs
split,,,,,
train,129834,2498,180,537,52
validation,27480,2499,551,621,11
test,24994,2500,635,698,10
embargo,4997,2499,544,628,2


## 5. Kiểm tra độc lập và ranh giới thời gian

Đối chiếu 300 household–snapshot train chọn ngẫu nhiên, trực tiếp từ basket, gồm cả target và các cửa sổ lịch sử. Kiểm tra ranh giới bằng ví dụ tổng hợp không phụ thuộc dữ liệu thực. Hai ngày 278/643 được giữ; snapshot có cửa sổ nhãn chạm hai ngày được đánh dấu để hỗ trợ phân tích độ nhạy sau này, không mặc định là nhãn sai.

In [5]:
def count_in_window(days, start, end):
    return ((days > start) & (days <= end)).sum()
boundary_days = np.array([10, 11, 17, 18])
assert count_in_window(boundary_days, 10, 17) == 2  # 11, 17; không gồm 10 hoặc 18
sample = panel.loc[panel.split.eq('train')].sample(300, random_state=42)
basket_by_household = {h: g for h,g in baskets.groupby('household_key', sort=False)}
for row in sample.itertuples(index=False):
    source = basket_by_household[row.household_key]
    future = source[(source.DAY > row.snapshot_day) & (source.DAY <= row.snapshot_day+HORIZON)]
    assert row.target_any_basket == int(not future.empty)
    for w in WINDOWS:
        history = source[(source.DAY > row.snapshot_day-w) & (source.DAY <= row.snapshot_day)]
        assert getattr(row, f'baskets_{w}d') == len(history)
        assert getattr(row, f'active_days_{w}d') == history.DAY.nunique()
        assert getattr(row, f'line_items_{w}d') == history.line_items.sum()
        assert np.isclose(getattr(row, f'sales_{w}d'), history.sales.sum(), atol=1e-6)
assert np.isfinite(panel.select_dtypes(include='number').to_numpy()).all()
panel['label_window_touches_anomaly'] = panel.snapshot_day.map(
    lambda s: any(s < d <= s+HORIZON for d in (278,643)))
print('PASS: ranh giới thời gian, 300 đối chiếu train, giá trị hữu hạn và tính duy nhất.')
display(panel.loc[panel.label_window_touches_anomaly].groupby(['split','snapshot_day']).size().rename('household_snapshot').to_frame())

PASS: ranh giới thời gian, 300 đối chiếu train, giá trị hữu hạn và tính duy nhất.


,,household_snapshot
split,snapshot_day,
test,642,2499
train,271,2496


### 5.1. Độ dài quan sát và cửa sổ lịch sử có ngày bất thường

`observed_tenure_days` là thời gian tính từ basket đầu tiên đã ghi nhận. Ta báo cáo độ phủ và tỷ lệ target trên **train** theo số ngày quan sát tính cả ngày đầu; không xem khoảng trước lần đầu xuất hiện là lịch sử không mua. Đủ 180 ngày từ lần đầu không chứng minh nguồn hoàn chỉnh.

Cờ cửa sổ lịch sử đánh dấu cutoff có `(s-w,s]` chứa ngày 278 hoặc 643. Cờ phản ánh khả năng bị ảnh hưởng, không xác nhận thiếu dữ liệu hay xác định mức sai lệch.

In [6]:
panel['observed_days_inclusive'] = panel.observed_tenure_days + 1
train_audit = panel.loc[panel.split.eq('train')].copy()
train_audit['history_group'] = np.where(train_audit.observed_days_inclusive < 180,
                                        'Dưới 180 ngày', 'Từ 180 ngày')
history_summary = (train_audit.groupby('history_group')
                   .agg(household_snapshots=('household_key','size'),
                        unique_households=('household_key','nunique'),
                        target_rate=('target_any_basket','mean')))
display(history_summary)
for w in (90, 180):
    panel[f'history_{w}d_touches_anomaly'] = panel.snapshot_day.map(
        lambda s: any(s-w < d <= s for d in (278, 643)))
    counts = panel.loc[panel.split.eq('train')].groupby(f'history_{w}d_touches_anomaly').agg(
        household_snapshots=('household_key','size'), cutoffs=('snapshot_day','nunique'))
    print(f'Cửa sổ lịch sử {w} ngày — tập train:')
    display(counts)
panel.drop(columns=['observed_days_inclusive'], inplace=True)

,household_snapshots,unique_households,target_rate
history_group,,,
Dưới 180 ngày,24030,2483,0.519767
Từ 180 ngày,105804,2497,0.529791


Cửa sổ lịch sử 90 ngày — tập train:


,household_snapshots,cutoffs
history_90d_touches_anomaly,,
False,97373,39
True,32461,13


Cửa sổ lịch sử 180 ngày — tập train:


,household_snapshots,cutoffs
history_180d_touches_anomaly,,
False,64909,26
True,64925,26


Các nhóm trên có thể chứa cùng household tại nhiều cutoff. Tỷ lệ target khác nhau không chứng minh hiệu ứng của độ dài quan sát; đây là cơ sở để báo cáo kết quả mô hình theo nhóm sau này, bên cạnh đánh giá toàn bộ tập đủ điều kiện.

## 6. Xuất dữ liệu và bàn giao

Mỗi file chỉ chứa một tập thời gian; không xuất embargo để huấn luyện. Cờ `label_window_touches_anomaly`, `household_key`, `snapshot_day`, `split` và `target_any_basket` là metadata/nhãn, không phải biến đầu vào mặc định. Ảnh hưởng có thể còn tồn tại trong các cửa sổ lịch sử; cờ này chỉ đánh dấu cửa sổ nhãn.

Không fit imputer, scaler hay bộ chọn biến trên toàn panel. Nếu cần các phép biến đổi học từ dữ liệu, chỉ fit trên train trong giai đoạn modeling.

In [7]:
OUT.mkdir(parents=True, exist_ok=True)
excluded = {'household_key','snapshot_day','split','target_any_basket','label_window_touches_anomaly','history_90d_touches_anomaly','history_180d_touches_anomaly'}
feature_columns = [c for c in panel.columns if c not in excluded]
for split in ['train','validation','test']:
    panel.loc[panel.split.eq(split)].to_parquet(OUT / f'{split}.parquet', index=False)
manifest = {'target': 'target_any_basket', 'horizon_days': HORIZON,
            'history_windows_days': list(WINDOWS), 'features': feature_columns,
            'metadata': sorted(excluded - {'target_any_basket'}),
            'embargo_cutoffs': [544,628],
            'split_boundaries': {'train':[180,537], 'validation':[551,621], 'test':[635,698]},
            'test_previously_inspected': True,
            'notes': ['Demographic and marketing excluded: historical availability unverified.',
                      'QUANTITY excluded; anomalies retained.',
                      'Anomaly flags mark label, 90-day and 180-day history windows.']}
(OUT / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
for split in ['train','validation','test']:
    saved = pd.read_parquet(OUT / f'{split}.parquet')
    assert len(saved) == int(panel.split.eq(split).sum())
    assert saved.split.eq(split).all() and not saved.duplicated(['household_key','snapshot_day']).any()
print('Đã xuất:', OUT, '| Số biến đầu vào:', len(feature_columns))
# Hợp đồng dùng cho giai đoạn modeling: tách feature khỏi metadata.
import sys
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from propensity_v2.prepared_data import load_prepared_split
for split in ['train','validation','test']:
    checked, checked_features = load_prepared_split(OUT, split)
    assert checked_features == feature_columns
print('PASS: hợp đồng dữ liệu và danh sách biến đầu vào của từng tập.')

,household_snapshots,unique_households,target_rate
history_group,,,
Dưới 180 ngày,24030,2483,0.519767
Từ 180 ngày,105804,2497,0.529791


Cửa sổ lịch sử 90 ngày — tập train:


,household_snapshots,cutoffs
history_90d_touches_anomaly,,
False,97373,39
True,32461,13


Cửa sổ lịch sử 180 ngày — tập train:


,household_snapshots,cutoffs
history_180d_touches_anomaly,,
False,64909,26
True,64925,26


**Bàn giao:** Ba bảng đã được kiểm tra ranh giới thời gian và đối chiếu mẫu từ basket nguồn. Những giới hạn về thời điểm sẵn có của dữ liệu, hai ngày bất thường và trạng thái đã xem test cần được giữ trong báo cáo đánh giá.

Pipeline V2 cũ có bộ biến khác và dùng demographic; lệnh train cũ được chặn mặc định để tránh nhầm hợp đồng. Bộ Parquet này được xác thực bởi `propensity_v2.prepared_data.load_prepared_split`; việc tích hợp huấn luyện với hợp đồng mới thuộc giai đoạn modeling, chưa được thực hiện ở đây.